# Software Directories Web scrapping

**Pipeline (actual steps in this notebook):**
1. **RSD & HELMHOLTZ scraping** → collect software pages and extract GitHub repository URLs
2. **Repository selection & deduplication** → validate URLs and select a primary repo per project/org
3. **Metadata extraction** → fetch README, repository info and common metadata files (licenses, package files) and save to `data/raw/`
4. **Save results & analysis** → write combined and filtered repositories info CSVs

## 📋 Step 1: Import Required Libraries

In [33]:
# Install necessary packages (uncomment if needed)
! pip install requests beautifulsoup4 pandas selenium webdriver-manager cffconvert tqdm scikit-learn --quiet

import requests
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
import time
import re
import json
import os, base64
import numpy as np
from pathlib import Path
import random
import shutil
import subprocess
from collections import defaultdict, Counter
from datetime import datetime

# Selenium for advanced scraping
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("📚 All libraries imported successfully!")
print(f"🕐 Pipeline started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
📚 All libraries imported successfully!
🕐 Pipeline started at: 2025-10-31 06:24:48


## ⚙️ Step 2: Configuration and Setup

In [ ]:
# Configure Selenium driver
chrome_options = Options()
chrome_options.add_argument("--headless=new")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=chrome_options)

# URLs and configuration
BASE_URL = "https://research-software-directory.org"
LIST_URL = f"{BASE_URL}/software?page=1&rows=1000"
HELMHOLTZ_URL = "https://helmholtz.software"
HELMHOLTZ_LIST_URL = f"{HELMHOLTZ_URL}/software?page=1&rows=415"

# GitHub API configuration (IMPORTANT: Add your token here)
GITHUB_TOKEN = "YOUR_TOKEN_HERE"  # ⚠️ REPLACE WITH YOUR TOKEN
HEADERS = {
    "Authorization": f"token {GITHUB_TOKEN}",
    "Accept": "application/vnd.github.v3+json"
}

# Directory configuration
base_dir = Path("../data/raw")

print("⚙️ Configuration completed!")
print(f"📁 Base directory: {base_dir}")
print(f"🔗 RSD URL: {LIST_URL}")
print(f"🎯 FULL EXTRACTION MODE: Up to 1000 repositories will be processed")

⚙️ Configuration completed!
📁 Base directory: ../data/raw
🔗 RSD URL: https://research-software-directory.org/software?page=1&rows=1000
🎯 FULL EXTRACTION MODE: Up to 1000 repositories will be processed


## 🕸️ Step 3: Extract GitHub Links from RSD

In [35]:
def get_software_entry_urls(list_url, base_url):
    """Get URLs of all software detail pages from the listing page"""
    print(f"🔍 Fetching software list from {list_url}")
    r = requests.get(list_url)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")

    software_links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/software/") and href != "/software":
            full_url = base_url + href
            software_links.append(full_url)

    software_links = sorted(set(software_links))
    print(f"✅ Found {len(software_links)} software detail page URLs")
    return software_links

def is_valid_github_repo_url(repo_url):
    """Validate if a GitHub URL follows the owner/repo pattern"""
    if not repo_url or "github.com" not in repo_url:
        return False
    
    # Pattern to match https://github.com/owner/repo (and optional trailing slash)
    pattern = r'^https://github\.com/([^/]+)/([^/]+)/?$'
    match = re.match(pattern, repo_url.rstrip('/'))
    
    if not match:
        return False
    
    owner, repo = match.groups()
    
    # Additional validation rules
    invalid_patterns = [
        # Common GitHub pages that aren't repositories
        r'^(features|marketplace|actions|orgs|settings|notifications|issues|pulls|discussions)$',
        # File extensions that shouldn't be repo names
        r'\.(git|zip|tar|gz)$',
        # Empty or invalid characters
        r'^$|^\.$|^\.\.$'
    ]
    
    for pattern in invalid_patterns:
        if re.search(pattern, owner, re.IGNORECASE) or re.search(pattern, repo, re.IGNORECASE):
            return False
    
    # Check for valid characters (GitHub allows alphanumeric, hyphens, underscores, dots)
    if not re.match(r'^[a-zA-Z0-9._-]+$', owner) or not re.match(r'^[a-zA-Z0-9._-]+$', repo):
        return False
    
    return True

def is_main_repo(repo_url):
    """Determine if a GitHub URL is a main repository"""
    if not is_valid_github_repo_url(repo_url):
        return False
        
    excluded_patterns = [
        r'/features/', r'/marketplace/', r'/actions/', r'/orgs/', r'/-/',
        r'/blob/', r'/tree/', r'/releases/', r'/issues/', r'/pull/',
        r'/wiki/', r'\.git$', r'/commit/', r'/compare/', r'/archive/',
        r'/zipball/', r'/tarball/', r'/fork'
    ]
    
    for pattern in excluded_patterns:
        if re.search(pattern, repo_url, re.IGNORECASE):
            return False
    return True

def select_primary_repo(repos):
    """Select the main repository from a list"""
    if not repos:
        return None
    
    # First filter for valid GitHub repo URLs
    valid_repos = [repo for repo in repos if is_valid_github_repo_url(repo)]
    if not valid_repos:
        return None
    
    # Then filter for main repos
    main_repos = [repo for repo in valid_repos if is_main_repo(repo)]
    if not main_repos:
        # If no main repos found, return the first valid one
        return valid_repos[0]
    if len(main_repos) == 1:
        return main_repos[0]
    
    # Prefer shortest (often the main one)
    return min(main_repos, key=len)

def extract_github_org(repo_url):
    """Extract organization/user name from GitHub URL"""
    if not repo_url or not is_valid_github_repo_url(repo_url):
        return None
    
    pattern = r'https://github\.com/([^/]+)/([^/]+)/?$'
    match = re.match(pattern, repo_url.rstrip('/'))
    return match.group(1) if match else None

def extract_repo_name(repo_url):
    """Extract repository name from GitHub URL"""
    if not repo_url or not is_valid_github_repo_url(repo_url):
        return None
    
    pattern = r'https://github\.com/([^/]+)/([^/]+)/?$'
    match = re.match(pattern, repo_url.rstrip('/'))
    return match.group(2) if match else None

def extract_github_repos_from_page(page_url):
    """Extract GitHub URLs from a software detail page"""
    try:
        driver.get(page_url)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        
        page_source = driver.page_source
        soup = BeautifulSoup(page_source, "html.parser")
        
        # Extract project name
        title_elem = soup.find("h1") or soup.find("title")
        project_name = title_elem.get_text(strip=True) if title_elem else "Unknown"
        
        # Find all GitHub links
        github_links = []
        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "github.com" in href and href.startswith("https://github.com/"):
                # Only add if it's a valid repository URL
                if is_valid_github_repo_url(href):
                    github_links.append(href)
        
        # Deduplication and cleanup
        github_links = list(set(github_links))
        
        # Select main repository
        main_repo = select_primary_repo(github_links)
        
        # Only return data if we have a valid main repository
        if not main_repo:
            return None
        
        return {
            "software_name": project_name,
            "software_url": page_url,
            "repository_url": main_repo,
            "all_github_links": github_links,
            "organization": extract_github_org(main_repo),
            "repository_name": extract_repo_name(main_repo),
            "owner_repo_pattern": f"{extract_github_org(main_repo)}/{extract_repo_name(main_repo)}"
        }
        
    except Exception as e:
        print(f"❌ Error processing {page_url}: {e}")
        return None

def filter_one_repo_per_org(repo_data):
    """Keep only 1 repository per organization - ensures no organization duplication"""
    seen_orgs = set()
    filtered_repos = []
    
    # Sort by organization and project name for consistent selection
    sorted_repos = sorted(repo_data, key=lambda x: (x.get('organization') or 'zzz', x['software_name']))
    
    for repo in sorted_repos:
        org = repo.get('organization')
        
        if not org or org not in seen_orgs:
            filtered_repos.append(repo)
            if org:
                seen_orgs.add(org)
    
    return filtered_repos

def validate_final_results(repo_data):
    """Final validation to ensure all repositories follow owner/repo pattern"""
    valid_repos = []
    invalid_repos = []
    
    for repo in repo_data:
        if is_valid_github_repo_url(repo['repository_url']):
            valid_repos.append(repo)
        else:
            invalid_repos.append(repo)
    
    if invalid_repos:
        print(f"🔧 Filtered out {len(invalid_repos)} invalid repository URLs")
    
    return valid_repos

# Execute GitHub links extraction from both RSD and HELMHOLTZ
print("🚀 Starting GitHub links extraction from RSD and HELMHOLTZ...")

# Get all software URLs from RSD
print("\n📋 Extracting from RSD...")
rsd_software_urls = get_software_entry_urls(LIST_URL, BASE_URL)

# Get all software URLs from HELMHOLTZ
print("\n📋 Extracting from HELMHOLTZ...")
helmholtz_software_urls = get_software_entry_urls(HELMHOLTZ_LIST_URL, HELMHOLTZ_URL)

# Combine all URLs
all_software_urls = rsd_software_urls + helmholtz_software_urls
print(f"\n📊 Total software URLs: {len(all_software_urls)} (RSD: {len(rsd_software_urls)}, HELMHOLTZ: {len(helmholtz_software_urls)})")

# Extract GitHub repositories
all_repo_data = []
failed_urls = []

for i, url in enumerate(tqdm(all_software_urls, desc="🔍 Extracting GitHub repos")):
    repo_data = extract_github_repos_from_page(url)
    
    if repo_data and repo_data["repository_url"]:
        # Add source information
        repo_data["source"] = "RSD" if url.startswith(BASE_URL) else "HELMHOLTZ"
        all_repo_data.append(repo_data)
    else:
        failed_urls.append(url)
    
    # Pause to avoid overload
    if i % 50 == 0 and i > 0:
        time.sleep(2)

print(f"\n✅ Initial extraction completed!")
print(f"📊 Total repositories found: {len(all_repo_data)}")

# Validate all repositories have proper owner/repo pattern
print(f"\n🔧 Validating repository URLs...")
all_repo_data = validate_final_results(all_repo_data)

# Apply organization filtering (1 repo per org)
print(f"\n🔧 Applying organization filtering (1 repo per organization)...")
filtered_repo_data = filter_one_repo_per_org(all_repo_data)

print(f"\n📊 After filtering:")
print(f"   Repositories before: {len(all_repo_data)}")
print(f"   Repositories after: {len(filtered_repo_data)}")
print(f"   Reduction: {len(all_repo_data) - len(filtered_repo_data)} repositories")
print(f"   Retention rate: {len(filtered_repo_data)/len(all_repo_data)*100:.1f}%")

# Create DataFrames
df_all_repos = pd.DataFrame(all_repo_data)
df_filtered_repos = pd.DataFrame(filtered_repo_data)

print(f"❌ Failed pages: {len(failed_urls)}")
print(f"📈 Success rate: {len(df_filtered_repos)/(len(all_software_urls))*100:.1f}%")

# Save results
csv_output_all = "combined_github_repositories_all.csv"
csv_output_filtered = "combined_github_repositories_filtered.csv"

df_all_repos.to_csv(csv_output_all, index=False)
df_filtered_repos.to_csv(csv_output_filtered, index=False)

print(f"\n💾 Data saved to:")
print(f"   All repositories: {csv_output_all}")
print(f"   Filtered repositories (max 1 per org): {csv_output_filtered}")

# Show source distribution
source_counts = df_filtered_repos['source'].value_counts()
print(f"\n📊 Source distribution in filtered dataset:")
for source, count in source_counts.items():
    print(f"   {source}: {count} repositories")

# Show validation summary
print(f"\n🔍 Repository URL validation summary:")
print(f"   All repositories follow owner/repo pattern: ✅")
print(f"   Sample owner/repo patterns:")
for i, row in df_filtered_repos.head().iterrows():
    print(f"   - {row['owner_repo_pattern']}")

# Close driver
driver.quit()

# Show preview of filtered results
print(f"\n🔍 Preview of filtered dataset (first 5 projects):")
for i, row in df_filtered_repos.head().iterrows():
    print(f"{i+1}. {row['software_name'][:50]}...")
    print(f"   Repository: {row['repository_url']}")
    print(f"   Owner/Repo: {row['owner_repo_pattern']}")
    print(f"   Organization: {row['organization']}")
    print(f"   Source: {row['source']}")
    print()

🚀 Starting GitHub links extraction from RSD and HELMHOLTZ...

📋 Extracting from RSD...
🔍 Fetching software list from https://research-software-directory.org/software?page=1&rows=1000
✅ Found 1000 software detail page URLs

📋 Extracting from HELMHOLTZ...
🔍 Fetching software list from https://helmholtz.software/software?page=1&rows=415
✅ Found 415 software detail page URLs

📊 Total software URLs: 1415 (RSD: 1000, HELMHOLTZ: 415)


🔍 Extracting GitHub repos: 100%|██████████| 1415/1415 [10:42<00:00,  2.20it/s]  



✅ Initial extraction completed!
📊 Total repositories found: 806

🔧 Validating repository URLs...

🔧 Applying organization filtering (1 repo per organization)...

📊 After filtering:
   Repositories before: 806
   Repositories after: 501
   Reduction: 305 repositories
   Retention rate: 62.2%
❌ Failed pages: 609
📈 Success rate: 35.4%

💾 Data saved to:
   All repositories: combined_github_repositories_all.csv
   Filtered repositories (max 1 per org): combined_github_repositories_filtered.csv

📊 Source distribution in filtered dataset:
   RSD: 311 repositories
   HELMHOLTZ: 190 repositories

🔍 Repository URL validation summary:
   All repositories follow owner/repo pattern: ✅
   Sample owner/repo patterns:
   - 3D-e-Chem/3D-e-Chem-VM
   - 4C-multiphysics/4C
   - 524D/compareMS2
   - AA-ALERT/AMBER
   - ADAH-EviDENce/evidence

🔍 Preview of filtered dataset (first 5 projects):
1. 3D-e-Chem Virtual Machine...
   Repository: https://github.com/3D-e-Chem/3D-e-Chem-VM
   Owner/Repo: 3D-e-Chem/3

In [36]:
# --- IGNORE ---
# ONLY RUN THIS CELL IF YOU WANT TO LOAD THE FILTERED REPOSITORIES AGAIN FROM THE CSV
# --- IGNORE ---
 
# load filtered repositories for metadata extraction
df_filtered_repos = pd.read_csv("combined_github_repositories_filtered.csv")

## 🔍 Step 4: GitHub Metadata Extraction Functions

In [37]:
def extract_github_metadata(repo_url):
    """Extract metadata from GitHub repository"""
    try:
        # Parse owner/repo from URL
        parts = repo_url.replace('https://github.com/', '').split('/')
        if len(parts) < 2:
            return None
        
        owner, repo = parts[0], parts[1]
        api_url = f'https://api.github.com/repos/{owner}/{repo}'
        
        # Get repository info
        response = requests.get(api_url, headers=HEADERS)
        if response.status_code != 200:
            return None
        
        repo_data = response.json()
        
        # Get README
        readme_api_url = f'{api_url}/readme'
        readme_response = requests.get(readme_api_url, headers=HEADERS)
        readme_html_url = None
        if readme_response.status_code == 200:
            readme_data = readme_response.json()
            readme_html_url = readme_data.get('html_url')

        # Get programming languages
        languages = []
        languages_api_url = f'{api_url}/languages'
        languages_response = requests.get(languages_api_url, headers=HEADERS)
        if languages_response.status_code == 200:
            languages_data = languages_response.json()
            languages = list(languages_data.keys()) 
        
        # Extract metadata
        metadata = {
            'codeRepository': repo_data.get('html_url'),
            'programmingLanguage': repo_data.get('language'),
            'programmingLanguages': languages,
            'downloadUrl': repo_data.get('archive_url', '').replace('{archive_format}{/ref}', 'zipball/main'),
            'author': repo_data.get('owner', {}).get('login'),
            'dateCreated': repo_data.get('created_at'),
            'dateModified': repo_data.get('updated_at'),
            'keywords': repo_data.get('topics', []),
            'license': repo_data.get('license', {}).get('url') if repo_data.get('license') else None,
            'description': repo_data.get('description'),
            'identifier': repo_data.get('id'),
            'name': repo_data.get('full_name'),
            'issueTracker': repo_data.get('issues_url', '').replace('{/number}', ''),
            'readme': readme_html_url
        }
        
        return metadata
        
    except Exception:
        return None

def save_project_files(project_dir, metadata, repo_url, headers=None):
    """Save extracted metadata and files to project directory (case-insensitive)"""
    if headers is None:
        headers = HEADERS
    
    project_dir = Path(project_dir)
    project_dir.mkdir(parents=True, exist_ok=True)
    
    # Save repository info
    with open(project_dir / 'repo_info.json', 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    # Extract comprehensive research software metadata files
    try:
        parts = repo_url.rstrip('/').replace('https://github.com/', '').split('/')
        owner, repo = parts[0], parts[1]
        
        # Define all metadata files to extract (lowercase for comparison)
        metadata_files = {
            'citation.cff': 'CITATION.cff',
            'codemeta.json': 'codemeta.json',
            '.zenodo.json': '.zenodo.json',
            'package.json': 'package.json',
            'pyproject.toml': 'pyproject.toml',
            'setup.py': 'setup.py',
            'cargo.toml': 'Cargo.toml',
            'pom.xml': 'pom.xml',
            'composer.json': 'composer.json',
        }
        
        # Get repository contents to find files case-insensitively
        contents_url = f'https://api.github.com/repos/{owner}/{repo}/contents/'
        response = requests.get(contents_url, headers=headers)
        
        extracted_files = []
        
        if response.status_code == 200:
            repo_contents = response.json()
            # Create a case-insensitive lookup
            file_lookup = {item['name'].lower(): item['name'] for item in repo_contents if item['type'] == 'file'}
            
            for file_key, preferred_name in metadata_files.items():
                # Check if file exists (case-insensitive)
                if file_key in file_lookup:
                    actual_filename = file_lookup[file_key]
                    file_url = f'https://api.github.com/repos/{owner}/{repo}/contents/{actual_filename}'
                    file_response = requests.get(file_url, headers=headers)
                    
                    if file_response.status_code == 200:
                        file_data = file_response.json()
                        if file_data.get('content'):
                            content = base64.b64decode(file_data['content']).decode('utf-8')
                            
                            # Save with actual filename from repo
                            file_path = project_dir / actual_filename
                            file_path.parent.mkdir(parents=True, exist_ok=True)
                            
                            with open(file_path, 'w', encoding='utf-8') as f:
                                f.write(content)
                            extracted_files.append(actual_filename)
            
            # Extract all README files (case-insensitive, starting with 'readme')
            for item in repo_contents:
                if item['type'] == 'file':
                    lower_name = item['name'].lower()
                    if lower_name.startswith('readme'):
                        actual_filename = item['name']
                        file_url = f'https://api.github.com/repos/{owner}/{repo}/contents/{actual_filename}'
                        file_response = requests.get(file_url, headers=headers)
                        
                        if file_response.status_code == 200:
                            file_data = file_response.json()
                            if file_data.get('content'):
                                content = base64.b64decode(file_data['content']).decode('utf-8')
                                
                                # Save with actual filename from repo
                                file_path = project_dir / actual_filename
                                file_path.parent.mkdir(parents=True, exist_ok=True)
                                
                                with open(file_path, 'w', encoding='utf-8') as f:
                                    f.write(content)
                                extracted_files.append(actual_filename)
            
            # Extract author files (case-insensitive, named 'authors' or starting with 'authors')
            for item in repo_contents:
                if item['type'] == 'file':
                    lower_name = item['name'].lower()
                    if lower_name == 'authors' or lower_name.startswith('authors'):
                        actual_filename = item['name']
                        file_url = f'https://api.github.com/repos/{owner}/{repo}/contents/{actual_filename}'
                        file_response = requests.get(file_url, headers=headers)
                        
                        if file_response.status_code == 200:
                            file_data = file_response.json()
                            if file_data.get('content'):
                                content = base64.b64decode(file_data['content']).decode('utf-8')
                                
                                # Save with actual filename from repo
                                file_path = project_dir / actual_filename
                                file_path.parent.mkdir(parents=True, exist_ok=True)
                                
                                with open(file_path, 'w', encoding='utf-8') as f:
                                    f.write(content)
                                extracted_files.append(actual_filename)
        
        with open(project_dir / 'extracted_files.json', 'w', encoding='utf-8') as f:
            json.dump({'extracted_files': extracted_files}, f, indent=2)
            
    except Exception as e:
        with open(project_dir / 'extraction_errors.log', 'w') as f:
            f.write(f'Error during file extraction: {str(e)}')

print('🔧 Essential GitHub metadata extraction functions defined!')

🔧 Essential GitHub metadata extraction functions defined!


## 🚀 Step 5: Extract Metadata for All Repositories

In [41]:
# Create base directory
base_dir.mkdir(exist_ok=True)

print(f'🚀 Starting metadata extraction for {len(df_filtered_repos)} repositories...')

successful_extractions = 0
failed_extractions = 0

for idx, row in tqdm(df_filtered_repos.iterrows(), total=len(df_filtered_repos), desc='📥 Extracting metadata'):
    try:
        repo_url = row['repository_url']
        project_name = row['software_name']
        
        # Create safe directory name
        safe_name = re.sub(r'[^\w\-_\.]', '_', project_name)[:100]
        project_dir = base_dir / f'{idx:03d}_{safe_name}'
        
        # Extract metadata
        metadata = extract_github_metadata(repo_url)
        
        if metadata:
            save_project_files(project_dir, metadata, repo_url)
            successful_extractions += 1
        else:
            failed_extractions += 1
        
        # Rate limiting
        if idx % 50 == 0 and idx > 0:
            time.sleep(1)
            
    except Exception as e:
        failed_extractions += 1
        continue

print(f'\n✅ Metadata extraction completed!')
print(f'📊 Successful extractions: {successful_extractions}')
print(f'❌ Failed extractions: {failed_extractions}')
print(f'📈 Success rate: {successful_extractions/(successful_extractions+failed_extractions)*100:.1f}%')

# Verify created directories
created_dirs = [d for d in base_dir.iterdir() if d.is_dir()]
print(f'📁 Project directories created: {len(created_dirs)}')

🚀 Starting metadata extraction for 501 repositories...


📥 Extracting metadata: 100%|██████████| 501/501 [18:27<00:00,  2.21s/it]


✅ Metadata extraction completed!
📊 Successful extractions: 497
❌ Failed extractions: 4
📈 Success rate: 99.2%
📁 Project directories created: 497


## 📊 Step 6: Dataset Analysis and Statistics

In [42]:
def analyze_extracted_dataset(base_directory):
    """Analyze the extracted dataset and provide comprehensive statistics"""
    
    # Get all project directories
    project_dirs = [d for d in base_directory.iterdir() if d.is_dir()]
    total_repos = len(project_dirs)
    
    # Initialize counters
    language_counts = Counter()
    categorized_file_counts = Counter()
    readme_counts = Counter()
    author_counts = Counter()
    total_files_extracted = 0
    repos_with_files = 0
    repos_with_readme = 0
    repos_with_authors = 0
    
    print(f"🔍 Analyzing {total_repos} repositories...")
    
    for project_dir in tqdm(project_dirs, desc="📊 Analyzing projects"):
        # Load repo info for language detection
        repo_info_file = project_dir / 'repo_info.json'
        if repo_info_file.exists():
            try:
                with open(repo_info_file, 'r', encoding='utf-8') as f:
                    repo_data = json.load(f)
                language = repo_data.get('programmingLanguage', 'Unknown')
                if language:
                    language_counts[language] += 1
            except Exception:
                language_counts['Unknown'] += 1
        
        # Count extracted files (case-insensitive categorization)
        extracted_files_json = project_dir / 'extracted_files.json'
        if extracted_files_json.exists():
            try:
                with open(extracted_files_json, 'r', encoding='utf-8') as f:
                    extracted_data = json.load(f)
                files = extracted_data.get('extracted_files', [])
                if files:
                    repos_with_files += 1
                    total_files_extracted += len(files)
                    
                    has_readme = False
                    has_authors = False
                    
                    for file in files:
                        lower_name = file.lower()
                        
                        # Categorize README files (case-insensitive, starting with 'readme')
                        if lower_name.startswith('readme'):
                            categorized_file_counts['README files'] += 1
                            readme_counts[lower_name] += 1
                            has_readme = True
                        
                        # Categorize AUTHOR files (case-insensitive, starting with 'author' or 'contributors')
                        elif lower_name.startswith('author') or lower_name.startswith('contributor'):
                            categorized_file_counts['AUTHOR files'] += 1
                            author_counts[lower_name] += 1
                            has_authors = True
                        
                        # Specific metadata files (exact match after lower)
                        else:
                            categorized_file_counts[lower_name] += 1
                    
                    if has_readme:
                        repos_with_readme += 1
                    if has_authors:
                        repos_with_authors += 1
                        
            except Exception:
                pass
    
    # Combine into file_distribution with categories first
    file_distribution = dict(categorized_file_counts.most_common())
    
    return {
        'total_repositories': total_repos,
        'repos_with_metadata': repos_with_files,
        'total_files_extracted': total_files_extracted,
        'repos_with_readme': repos_with_readme,
        'repos_with_authors': repos_with_authors,
        'language_distribution': dict(language_counts.most_common()),
        'file_distribution': file_distribution,
        'readme_variants': dict(readme_counts.most_common()),
        'author_variants': dict(author_counts.most_common()),
        'extraction_rate': (repos_with_files / total_repos * 100) if total_repos > 0 else 0
    }

def print_analysis_report(stats):
    """Print comprehensive analysis report"""
    print("\n" + "=" * 60)
    print("📊 DATASET ANALYSIS REPORT")
    print("=" * 60)
    
    # Overview
    print(f"📈 OVERVIEW:")
    print(f"   Total Repositories: {stats['total_repositories']:,}")
    print(f"   Repositories with Metadata: {stats['repos_with_metadata']:,}")
    print(f"   Repositories with README: {stats['repos_with_readme']:,}")
    print(f"   Repositories with AUTHORS: {stats['repos_with_authors']:,}")
    print(f"   Total Files Extracted: {stats['total_files_extracted']:,}")
    print(f"   Extraction Success Rate: {stats['extraction_rate']:.1f}%")
    print(f"   Average Files per Repo: {stats['total_files_extracted']/stats['repos_with_metadata']:.1f}" if stats['repos_with_metadata'] > 0 else "   Average Files per Repo: 0")
    
    # Programming Languages
    print(f"\n🔤 PROGRAMMING LANGUAGES:")
    for i, (lang, count) in enumerate(list(stats['language_distribution'].items())[:10]):
        percentage = (count / stats['total_repositories']) * 100
        print(f"   {i+1:2d}. {lang:<15} {count:4d} ({percentage:5.1f}%)")
    
    # File Distribution (categorized)
    print(f"\n📁 EXTRACTED FILES DISTRIBUTION (CATEGORIZED):")
    total_cat_files = sum(stats['file_distribution'].values())
    for i, (category, count) in enumerate(stats['file_distribution'].items()):
        percentage = (count / total_cat_files) * 100 if total_cat_files > 0 else 0
        print(f"   {i+1:2d}. {category:<25} {count:4d} ({percentage:5.1f}%)")
    
    # README Variants
    if stats['readme_variants']:
        print(f"\n📄 README VARIANTS:")
        total_readmes = sum(stats['readme_variants'].values())
        for i, (variant, count) in enumerate(stats['readme_variants'].items()):
            percentage = (count / total_readmes) * 100 if total_readmes > 0 else 0
            print(f"   {i+1:2d}. {variant:<25} {count:4d} ({percentage:5.1f}%)")
    
    # AUTHOR Variants
    if stats['author_variants']:
        print(f"\n👥 AUTHOR VARIANTS:")
        total_authors = sum(stats['author_variants'].values())
        for i, (variant, count) in enumerate(stats['author_variants'].items()):
            percentage = (count / total_authors) * 100 if total_authors > 0 else 0
            print(f"   {i+1:2d}. {variant:<25} {count:4d} ({percentage:5.1f}%)")
    
    print("=" * 60)

# Execute analysis
print("🚀 Starting dataset analysis...")
analysis_stats = analyze_extracted_dataset(base_dir)
print_analysis_report(analysis_stats)

# Save analysis results
with open('dataset_analysis.json', 'w', encoding='utf-8') as f:
    json.dump(analysis_stats, f, indent=2, ensure_ascii=False)

print("💾 Analysis results saved to: dataset_analysis.json")

🚀 Starting dataset analysis...
🔍 Analyzing 497 repositories...


📊 Analyzing projects:   0%|          | 0/497 [00:00<?, ?it/s]

📊 Analyzing projects: 100%|██████████| 497/497 [00:00<00:00, 3014.55it/s]


📊 DATASET ANALYSIS REPORT
📈 OVERVIEW:
   Total Repositories: 497
   Repositories with Metadata: 495
   Repositories with README: 494
   Repositories with AUTHORS: 43
   Total Files Extracted: 1,213
   Extraction Success Rate: 99.6%
   Average Files per Repo: 2.5

🔤 PROGRAMMING LANGUAGES:
    1. Python           182 ( 36.6%)
    2. Jupyter Notebook   60 ( 12.1%)
    3. C++               59 ( 11.9%)
    4. Java              30 (  6.0%)
    5. R                 23 (  4.6%)
    6. JavaScript        21 (  4.2%)
    7. TypeScript        16 (  3.2%)
    8. Julia             14 (  2.8%)
    9. C                 12 (  2.4%)
   10. MATLAB            11 (  2.2%)

📁 EXTRACTED FILES DISTRIBUTION (CATEGORIZED):
    1. README files               545 ( 44.9%)
    2. citation.cff               221 ( 18.2%)
    3. pyproject.toml             146 ( 12.0%)
    4. setup.py                   127 ( 10.5%)
    5. .zenodo.json                62 (  5.1%)
    6. AUTHOR files                43 (  3.5%)
    7. pac